# Conflict Risk Model Based on GDELT OSINT News via Google BigQuery

**An open-source intelligence pipeline that quantifies escalation pressure between Israel and seven regional entities using GDELT 2.0 Events and Global Knowledge Graph data, queried live from Google BigQuery.**

Anchor country: **Israel**. Target dyads: **Egypt, Turkey, Jordan, Gaza, West Bank, Iran, Lebanon**.

Every date window in this notebook rolls forward from `date.today()`. Re-run it any day, any year - the analysis windows update automatically.


## 1. What is GDELT?

[**GDELT** (Global Database of Events, Language, and Tone)](https://www.gdeltproject.org/) is an open-source project that monitors print, broadcast, and web news media from nearly every country in over 100 languages, and codes every observed event into a structured machine-readable record. It is updated every 15 minutes.

Two GDELT tables drive this notebook:

<table>
<tr>
<td width="50%" align="center"><b>Events 2.0</b><br/><i>Who did what to whom, where, when</i><br/><br/><img src="assets/gdelt_events_map.jpg" width="460" alt="GDELT Events over NASA night lights"/><br/><sub>One row per coded event. CAMEO action codes, two actors, geographic coordinates, GoldsteinScale severity (-10 destabilising / +10 stabilising), AvgTone of coverage, NumMentions / NumSources / NumArticles coverage volume, SOURCEURL link to the news article.</sub></td>

<td width="50%" align="center"><b>Global Knowledge Graph 2.0</b><br/><i>What the news is talking about, and how</i><br/><br/><img src="assets/gdelt_gkg_graph.jpg" width="460" alt="GDELT Global Knowledge Graph network visualization"/><br/><sub>One row per news document. V2Themes (thousands of theme tags), V2Tone (7-part sentiment vector), V2Persons / V2Organizations / V2Locations named entities, GCAM cross-lingual emotion measures, plus the document URL.</sub></td>
</tr>
</table>

### Research idea

If a country is *about to escalate* with Israel - a border incident, a military drill, weapons procurement, hostile rhetoric - the world's media writes about it before the kinetic event happens. GDELT codes every one of those news items, every 15 minutes, globally. We can detect the **pattern** of escalation by comparing the rate of hostile coverage in the last 7 days against the rate in the last 30 days, dyad by dyad.

### Scope

- **What this is**: an early-warning *signal* of escalation patterns visible in open-source media.
- **What this is NOT**: a probability of war. It does not predict the timing, location, or scale of any conflict event. It is a media-coverage acceleration index. See section 11 (Limitations) before using any number from this notebook for any real decision.

### Method, in one paragraph

1. Fetch the last 30 days of GDELT Events filtered to each `(Israel, X)` dyad and the last 30 days of GDELT GKG documents mentioning both Israel and X.
2. Score each dyad on a fixed weighted bundle of hostile signals: violent CAMEO codes (threaten, force posture, coerce, assault, fight, mass violence) + GKG themes (armed-conflict, weapons procurement, terrorism).
3. Rank dyads by severity-weighted recent volume (min-max scaled across the seven dyads).
4. Compute correlation and covariance between dyads' daily hostility series.
5. Combine into one interactive dashboard showing internal instability vs bilateral pressure.
6. Show the entity co-occurrence network from GKG to expose the named actors driving each dyad.

Official GDELT project: **<https://www.gdeltproject.org/>**.


## 2. Configure

All notebook-wide constants live in the cell below. Edit the constants here, re-run the cell, and every downstream cell picks up the new values.

In [ ]:
import os
import datetime
import textwrap
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style="white", context="notebook")
plt.rcParams["axes.grid"] = False
plt.rcParams["figure.dpi"] = 110

OUT_DIR = Path(".")

EVENTS_TABLE = "gdelt-bq.gdeltv2.events_partitioned"
GKG_TABLE    = "gdelt-bq.gdeltv2.gkg_partitioned"

# COUNTRIES dict.
#   fips  : 2-letter ActionGeo_CountryCode used in Events
#   cameo : 3-letter Actor1CountryCode / Actor2CountryCode used in Events
#   gkg   : 2-letter FIPS used inside the V2Locations field of GKG
#   geo_names : optional substring filter for ActionGeo_FullName
#               (needed when several entities share a FIPS code, e.g. Gaza vs West Bank)
COUNTRIES = {
    "Israel":    {"fips": "IS", "cameo": "ISR", "gkg": "IS", "geo_names": None, "is_anchor": True},
    "Egypt":     {"fips": "EG", "cameo": "EGY", "gkg": "EG", "geo_names": None, "is_anchor": False},
    "Turkey":    {"fips": "TU", "cameo": "TUR", "gkg": "TU", "geo_names": None, "is_anchor": False},
    "Jordan":    {"fips": "JO", "cameo": "JOR", "gkg": "JO", "geo_names": None, "is_anchor": False},
    "Iran":      {"fips": "IR", "cameo": "IRN", "gkg": "IR", "geo_names": None, "is_anchor": False},
    "Lebanon":   {"fips": "LE", "cameo": "LBN", "gkg": "LE", "geo_names": None, "is_anchor": False},
    "Gaza": {
        "fips": "GZ", "cameo": "PSE", "gkg": "GZ",
        "geo_names": ["Gaza", "Khan Yunis", "Rafah", "Beit Hanoun", "Jabalia", "Deir al-Balah"],
        "is_anchor": False,
    },
    "West Bank": {
        "fips": "WE", "cameo": "PSE", "gkg": "WE",
        "geo_names": ["West Bank", "Ramallah", "Hebron", "Nablus", "Bethlehem", "Jenin", "Jericho", "Tulkarm", "Qalqilya"],
        "is_anchor": False,
    },
}
ANCHOR  = next(name for name, meta in COUNTRIES.items() if meta["is_anchor"])
TARGETS = [name for name, meta in COUNTRIES.items() if not meta["is_anchor"]]

SIGNALS = {
    "internal": {
        "cameo_roots": ["14", "17", "18", "19", "20"],
        "gkg_themes": [
            "WATER_SECURITY", "ENV_WATER", "INFRASTRUCTURE_BAD_ROADS",
            "POWER_OUTAGE", "CYBER_ATTACK", "REBELLION",
            "CIVIL_RESISTANCE", "PROTEST", "GOV_REQUIRES_REFORM",
        ],
    },
    "bilateral": {
        "cameo_roots": ["13", "15", "17", "18", "19", "20"],
        "gkg_themes": [
            "ARMEDCONFLICT", "MIL_WEAPONS_PROCUREMENT",
            "MIL_SELF_IDENTIFIED", "TERROR",
        ],
    },
}

TODAY    = datetime.date.today()
W7_FROM  = TODAY - datetime.timedelta(days=7)
W30_FROM = TODAY - datetime.timedelta(days=30)

_settings = pd.DataFrame({
    "setting": [
        "Anchor",
        "Targets (n)",
        "Window  7d",
        "Window 30d",
        "Internal CAMEO signals",
        "Internal GKG themes",
        "Bilateral CAMEO signals",
        "Bilateral GKG themes",
        "Output directory",
    ],
    "value": [
        ANCHOR,
        f"{len(TARGETS)} : {', '.join(TARGETS)}",
        f"{W7_FROM} -> {TODAY}",
        f"{W30_FROM} -> {TODAY}",
        ", ".join(SIGNALS["internal"]["cameo_roots"]),
        ", ".join(SIGNALS["internal"]["gkg_themes"]),
        ", ".join(SIGNALS["bilateral"]["cameo_roots"]),
        ", ".join(SIGNALS["bilateral"]["gkg_themes"]),
        str(OUT_DIR.resolve()) if OUT_DIR.resolve() != Path.cwd() else "<current directory>",
    ],
})
display(_settings.style.hide(axis="index").set_properties(**{"text-align": "left"}))


## 3. Install dependencies and authenticate to BigQuery

The first code cell installs every Python package the notebook needs and then prints a version table. The second cell asks you to paste a GCP project ID and a short-lived BigQuery access token. Both are read with `getpass` so they never appear in notebook output.

In [ ]:
import importlib, importlib.metadata as _meta

%pip install -q google-cloud-bigquery google-cloud-bigquery-storage pandas numpy matplotlib seaborn wordcloud scikit-learn plotly networkx kaleido

_pkgs = ["google-cloud-bigquery", "google-cloud-bigquery-storage", "pandas", "numpy",
         "matplotlib", "seaborn", "wordcloud", "scikit-learn", "plotly", "networkx", "kaleido"]
_rows = []
for _p in _pkgs:
    try:
        _rows.append({"library": _p, "version": _meta.version(_p)})
    except _meta.PackageNotFoundError:
        _rows.append({"library": _p, "version": "NOT INSTALLED"})

_lib_table = pd.DataFrame(_rows)
display(_lib_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))


### 3.b Authenticate to BigQuery

Open a fresh terminal on your computer:

```
gcloud auth login              # one-time, opens a browser
gcloud auth print-access-token # prints a ya29... token, lasts ~1 hour
```

Copy the token, then run the cell below. The project ID and the token are both read with `getpass` (hidden input) and are never echoed back in any cell output.

In [ ]:
import os
import getpass
import datetime
from google.cloud import bigquery
from google.oauth2.credentials import Credentials

for _k in ("BQ_TOKEN", "GOOGLE_APPLICATION_CREDENTIALS"):
    os.environ.pop(_k, None)

_project_id = getpass.getpass("GCP project ID (hidden): ").strip()
_bq_token = getpass.getpass(
    'BQ access token from "gcloud auth print-access-token" (hidden): '
).strip()

if not _project_id or not _bq_token:
    raise RuntimeError("Both a project ID and an access token are required.")


class _StaticBearerCreds(Credentials):
    """Bearer-token credentials whose refresh() raises a clean actionable error
    instead of the cryptic google-auth traceback when the token expires."""
    def refresh(self, request):
        raise RuntimeError(
            "Your BigQuery access token expired or was rejected by Google.\n"
            "Tokens from `gcloud auth print-access-token` last about 1 hour.\n"
            "Re-run THIS cell with a freshly-generated token."
        )


_creds = _StaticBearerCreds(token=_bq_token)
_creds.expiry = (
    datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
    + datetime.timedelta(minutes=55)
)
bq = bigquery.Client(project=_project_id, credentials=_creds)
del _bq_token
del _project_id


def dry_run_mb(sql: str, job_config=None) -> float:
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    if job_config is not None and getattr(job_config, "query_parameters", None):
        cfg.query_parameters = job_config.query_parameters
    job = bq.query(sql, job_config=cfg)
    return job.total_bytes_processed / (1024 * 1024)


# Free smoke test - fails fast if the token is bad.
try:
    _ = bq.query("SELECT 1 AS ok",
                 job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    print("Auth OK. Token valid for ~1 hour. Re-run this cell if later cells start raising RefreshError.")
except Exception as _e:
    raise RuntimeError(
        "Auth smoke test FAILED: " + type(_e).__name__ + ": " + str(_e) + "\n"
        "Common causes:\n"
        "  1. Token expired between paste and now (tokens last about 1 hour).\n"
        "  2. Token generated from a different Google account than the one that owns the project.\n"
        "  3. Project ID typo."
    ) from _e


## 4. Explore the Events table

The first thing to do with any new data source is look at it. The cells below pull a small sample of GDELT Events and show its shape, columns, and basic distributions.

In [ ]:
import textwrap

_events_tbl = bq.get_table(EVENTS_TABLE)
print(f"Table: {EVENTS_TABLE}")
print(f"Rows  : {_events_tbl.num_rows:,}")
print(f"Size  : {_events_tbl.num_bytes / 1e9:.2f} GB")
print(f"Partition: {_events_tbl.time_partitioning}")

_fields_str = ", ".join(f"{f.name} ({f.field_type})" for f in _events_tbl.schema)
print(f"\nSchema fields ({len(_events_tbl.schema)}):")
print(textwrap.fill(_fields_str, width=120, initial_indent="  ", subsequent_indent="  "))


In [ ]:
_sample_sql = f"""
SELECT
  PARSE_DATE("%Y%m%d", CAST(SQLDATE AS STRING)) AS date,
  EventCode, EventRootCode,
  Actor1Name, Actor2Name,
  CAST(AvgTone        AS FLOAT64) AS AvgTone,
  CAST(GoldsteinScale AS FLOAT64) AS GoldsteinScale,
  CAST(NumMentions    AS INT64)   AS NumMentions,
  ActionGeo_FullName,
  CAST(ActionGeo_Lat  AS FLOAT64) AS ActionGeo_Lat,
  CAST(ActionGeo_Long AS FLOAT64) AS ActionGeo_Long,
  SOURCEURL
FROM `{EVENTS_TABLE}`
WHERE _PARTITIONTIME >= TIMESTAMP("{W7_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE("{W7_FROM}"), INTERVAL 1 DAY))
  AND ActionGeo_CountryCode IN ({", ".join(repr(m["fips"]) for m in COUNTRIES.values())})
LIMIT 5000
"""
dry_run_mb(_sample_sql)

events_sample = bq.query(_sample_sql).result().to_dataframe(
    create_bqstorage_client=True, progress_bar_type=None,
)
print(f"events_sample: {len(events_sample):,} rows x {events_sample.shape[1]} columns")
display(events_sample.head())


In [ ]:
print("--- dtypes ---")
print(events_sample.dtypes.to_string())

print("\n--- describe (numeric) ---")
display(events_sample.describe(include=[np.number]).T)

print("\n--- top CAMEO root codes (top 10) ---")
_root_counts = events_sample["EventRootCode"].fillna("?").value_counts().head(10)
print(", ".join(f"{r}: {n:,}" for r, n in _root_counts.items()))


In [ ]:
def _annotate_bars(ax, values, fmt="{:,.0f}", offset=2):
    for rect, val in zip(ax.patches, values):
        try:
            v = float(val)
        except Exception:
            continue
        if np.isnan(v):
            continue
        if abs(v) >= 10000:
            continue
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + offset,
                fmt.format(v), ha="center", va="bottom", fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.hist(events_sample["AvgTone"].dropna(), bins=60, color="#4C78A8", alpha=0.85)
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_title("AvgTone distribution")
ax.set_xlabel("AvgTone (negative = hostile coverage)")
ax.set_ylabel("events")
ax.grid(False)

ax = axes[0, 1]
ax.hist(events_sample["GoldsteinScale"].dropna(), bins=40, color="#E45756", alpha=0.85)
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_title("GoldsteinScale distribution")
ax.set_xlabel("Goldstein  (-10 destabilising / +10 stabilising)")
ax.set_ylabel("events")
ax.grid(False)

ax = axes[1, 0]
_hostile = {"13","14","15","17","18","19","20"}
_roots = (events_sample["EventRootCode"].fillna("?")
                 .value_counts().sort_index())
_colors = ["#E45756" if str(c) in _hostile else "#72B7B2" for c in _roots.index]
_bars = ax.bar(_roots.index.astype(str), _roots.values, color=_colors)
_annotate_bars(ax, _roots.values)
ax.set_title("CAMEO root code distribution (red = hostile)")
ax.set_xlabel("EventRootCode")
ax.set_ylabel("events")
ax.tick_params(axis="x", rotation=45)
ax.grid(False)

ax = axes[1, 1]
_nm = events_sample["NumMentions"].dropna()
_nm = _nm[_nm > 0]
ax.hist(np.log10(_nm), bins=40, color="#54A24B", alpha=0.85)
ax.set_title("NumMentions (log10)")
ax.set_xlabel("log10(NumMentions)")
ax.set_ylabel("events")
ax.grid(False)

fig.suptitle(f"GDELT Events EDA - sample from {W7_FROM} for one day", fontsize=13)
fig.tight_layout()
plt.show()


### 4.5 Top actors word cloud

Names of the most-amplified actors in events involving the eight target countries over the last 30 days, sized by `NumMentions`. Generic noise tokens (`GOVERNMENT`, `MILITARY`, `UNKNOWN`, etc.) are filtered out so named entities surface.

In [ ]:
from wordcloud import WordCloud
from collections import Counter

_actor_sql = f"""
SELECT Actor1Name, Actor2Name, CAST(NumMentions AS INT64) AS NumMentions
FROM `{EVENTS_TABLE}`
WHERE _PARTITIONTIME >= TIMESTAMP("{W30_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE("{TODAY}"), INTERVAL 1 DAY))
  AND ActionGeo_CountryCode IN ({", ".join(repr(m["fips"]) for m in COUNTRIES.values())})
  AND NumMentions > 0
  AND (Actor1Name IS NOT NULL OR Actor2Name IS NOT NULL)
"""
dry_run_mb(_actor_sql)

_actors_df = bq.query(_actor_sql).result().to_dataframe(
    create_bqstorage_client=True, progress_bar_type=None,
)
print(f"Pulled {len(_actors_df):,} actor rows for word cloud.")

_STOP = {
    "UNKNOWN", "GOVERNMENT", "OFFICIAL", "REBEL", "CIVILIAN", "MILITANT",
    "NONE", "", "INTERNATIONAL", "PARLIAMENT", "MINISTER", "MILITARY",
    "WORLD", "PEOPLE", "FOREIGN", "WOMAN", "MAN", "REGION", "GROUP",
    "POLICE", "PRESIDENT", "MEDIA", "CITIZEN", "WORKER", "EMPLOYEE",
}

_cnt = Counter()
for _col in ("Actor1Name", "Actor2Name"):
    _slice = _actors_df[[_col, "NumMentions"]].dropna()
    _slice = _slice[~_slice[_col].str.upper().isin(_STOP)]
    for _name, _n in zip(_slice[_col], _slice["NumMentions"].astype(int)):
        _cnt[str(_name).strip().upper()] += int(_n)

print(f"Unique actors after stop-word filter: {len(_cnt):,}")
print("Top 10 by amplified mentions: " + ", ".join(f"{n} ({v:,})" for n, v in _cnt.most_common(10)))

if not _cnt:
    print("No actor names available - word cloud skipped.")
else:
    _wc = WordCloud(
        width=1400, height=700, background_color="white",
        colormap="viridis", max_words=120, prefer_horizontal=0.9,
    ).generate_from_frequencies(dict(_cnt))

    fig, ax = plt.subplots(figsize=(13, 6.5))
    ax.imshow(_wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"Top actors across 8 target countries  ({W30_FROM} to {TODAY})", fontsize=12)
    fig.tight_layout()
    Path("assets").mkdir(exist_ok=True)
    fig.savefig("assets/viz_top_actors.png", dpi=140, bbox_inches="tight")
    plt.show()


## 5. Explore the GKG table

The Global Knowledge Graph is a document-level layer over the same news sources. Each row is a news document with extracted themes, named entities, and a seven-part tone vector. Below we look at its schema and the most-frequent themes in the last 30 days.

In [ ]:
_gkg_tbl = bq.get_table(GKG_TABLE)
print(f"Table: {GKG_TABLE}")
print(f"Rows : {_gkg_tbl.num_rows:,}")
print(f"Size : {_gkg_tbl.num_bytes / 1e9:.2f} GB")

_fields_str = ", ".join(f"{f.name} ({f.field_type})" for f in _gkg_tbl.schema)
print(f"\nSchema fields ({len(_gkg_tbl.schema)}):")
print(textwrap.fill(_fields_str, width=120, initial_indent="  ", subsequent_indent="  "))


In [ ]:
_themes_sql = f"""
WITH unnested AS (
  SELECT SPLIT(raw, ",")[OFFSET(0)] AS theme
  FROM `{GKG_TABLE}`,
       UNNEST(SPLIT(V2Themes, ";")) AS raw
  WHERE _PARTITIONTIME >= TIMESTAMP("{W30_FROM}")
    AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE("{TODAY}"), INTERVAL 1 DAY))
    AND raw != ""
)
SELECT theme, COUNT(*) AS n
FROM unnested
GROUP BY theme
ORDER BY n DESC
LIMIT 25
"""
dry_run_mb(_themes_sql)

_themes_df = bq.query(_themes_sql).result().to_dataframe(
    create_bqstorage_client=True, progress_bar_type=None,
)

print(f"Top 25 GKG themes ({W30_FROM} to {TODAY}):")
print(textwrap.fill(", ".join(f"{t} ({n:,})" for t, n in zip(_themes_df["theme"], _themes_df["n"])),
                     width=120, initial_indent="  ", subsequent_indent="  "))


## 6. See real news per country, before any analysis

Before running any scoring, look at the raw material. The cell below pulls 10 random events from the last 30 days for each of the 8 target countries, with the actors, CAMEO code, AvgTone, GoldsteinScale, and source URL. Click through a few of the URLs to convince yourself the data behind the analysis is real.

In [ ]:
_news_sql_parts = []
for _name, _meta in COUNTRIES.items():
    _filter = f"ActionGeo_CountryCode = \"{_meta['fips']}\""
    if _meta.get("geo_names"):
        _terms = " OR ".join(
            [f"REGEXP_CONTAINS(IFNULL(ActionGeo_FullName, \"\"), r\"(?i){_g}\")" for _g in _meta["geo_names"]]
        )
        _filter = f"({_filter}) AND ({_terms})"
    _news_sql_parts.append(f"""
SELECT
  \"{_name}\" AS country,
  PARSE_DATE(\"%Y%m%d\", CAST(SQLDATE AS STRING)) AS date,
  Actor1Name, Actor2Name,
  CAST(EventCode AS STRING) AS EventCode,
  CAST(AvgTone AS FLOAT64) AS AvgTone,
  CAST(GoldsteinScale AS FLOAT64) AS GoldsteinScale,
  SOURCEURL,
  RAND() AS _r
FROM `{EVENTS_TABLE}`
WHERE _PARTITIONTIME >= TIMESTAMP(\"{W30_FROM}\")
  AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE(\"{TODAY}\"), INTERVAL 1 DAY))
  AND {_filter}
  AND SOURCEURL IS NOT NULL
""")
_news_sql = "WITH pool AS (\n" + "UNION ALL".join(_news_sql_parts) + ")\n"
_news_sql += """
SELECT country, date, Actor1Name, Actor2Name, EventCode, AvgTone, GoldsteinScale, SOURCEURL
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY country ORDER BY _r) AS rn
  FROM pool
)
WHERE rn <= 10
ORDER BY country, date DESC
"""
dry_run_mb(_news_sql)

news_samples = bq.query(_news_sql).result().to_dataframe(
    create_bqstorage_client=True, progress_bar_type=None,
)
print(f"Pulled {len(news_samples):,} sample rows (target: 80).")
for _country in COUNTRIES.keys():
    _sub = news_samples[news_samples["country"] == _country].copy()
    if _sub.empty:
        print(f"\n=== {_country}: no events in window ===")
        continue
    print(f"\n=== {_country}  ({len(_sub)} samples) ===")
    display(_sub.drop(columns=["country"]).reset_index(drop=True))


## 7. Signal helper functions

Four reusable functions used by the scoring sections. They issue partition-pruned BigQuery jobs, return clean pandas DataFrames, and handle the Gaza vs West Bank disambiguation (both share FIPS `WE` but differ by `ActionGeo_FullName`).

| Helper | Returns | Used by |
|---|---|---|
| `fetch_events(country, from, to)` | event rows for a country | `daily_internal_signals` and the news sample fetcher |
| `fetch_gkg(country, from, to)` | per-day document counts + V2Tone + theme hit counts for a country | `daily_internal_signals` |
| `daily_internal_signals(country, from, to)` | one row per day with CAMEO + GKG + tone columns | §8 internal instability |
| `daily_bilateral_signals(target, from, to)` | one row per day for events linking Israel and the target | §9 bilateral pressure, §10 correlation |

In [ ]:
import json
from typing import Optional

_INTERNAL_CAMEO   = SIGNALS["internal"]["cameo_roots"]
_INTERNAL_THEMES  = SIGNALS["internal"]["gkg_themes"]
_BILATERAL_CAMEO  = SIGNALS["bilateral"]["cameo_roots"]
_BILATERAL_THEMES = SIGNALS["bilateral"]["gkg_themes"]


def _daily_index(date_from, date_to):
    return pd.date_range(start=date_from, end=date_to, freq="D").date


def _geo_clause(country_meta, table_alias=""):
    if not country_meta.get("geo_names"):
        return ""
    col = f"{table_alias}ActionGeo_FullName" if table_alias else "ActionGeo_FullName"
    parts = " OR ".join(
        [f"REGEXP_CONTAINS(IFNULL({col}, ''), r'(?i){g}')" for g in country_meta["geo_names"]]
    )
    return f" AND ({parts})"


def fetch_events(country_name: str, date_from, date_to) -> pd.DataFrame:
    """Pull GDELT Events that took place inside `country_name`."""
    meta = COUNTRIES[country_name]
    sql = f"""
    SELECT
      PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)) AS date,
      CAST(EventRootCode AS STRING) AS EventRootCode,
      Actor1CountryCode, Actor2CountryCode,
      Actor1Name, Actor2Name,
      CAST(AvgTone AS FLOAT64) AS AvgTone,
      CAST(GoldsteinScale AS FLOAT64) AS GoldsteinScale,
      CAST(NumMentions AS INT64) AS NumMentions,
      ActionGeo_FullName, SOURCEURL
    FROM `{EVENTS_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{date_from}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{date_to}'), INTERVAL 1 DAY))
      AND ActionGeo_CountryCode = '{meta["fips"]}'
      {_geo_clause(meta)}
    """
    dry_run_mb(sql)
    return bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )


def fetch_gkg(country_name: str, date_from, date_to) -> pd.DataFrame:
    """Pull per-day GKG document counts + tone + theme hits for one country."""
    meta = COUNTRIES[country_name]
    theme_cases = ",\n        ".join(
        [f"COUNTIF(REGEXP_CONTAINS(IFNULL(V2Themes, ''), r'\\b{t}\\b')) AS {t}"
         for t in _INTERNAL_THEMES]
    )
    sql = f"""
    SELECT
      DATE(_PARTITIONTIME) AS date,
      COUNT(*) AS n_docs,
      AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS tone,
      AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(1)] AS FLOAT64)) AS positive,
      AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(2)] AS FLOAT64)) AS negative,
      {theme_cases}
    FROM `{GKG_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{date_from}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{date_to}'), INTERVAL 1 DAY))
      AND V2Locations LIKE '%#{meta["gkg"]}#%'
    GROUP BY date
    ORDER BY date
    """
    dry_run_mb(sql)
    return bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )


def daily_internal_signals(country_name: str, date_from, date_to) -> pd.DataFrame:
    """Per-day country signal panel: CAMEO event counts + GKG theme counts + tone."""
    ev  = fetch_events(country_name, date_from, date_to)
    gkg = fetch_gkg(country_name, date_from, date_to)

    idx = _daily_index(date_from, date_to)
    out = pd.DataFrame(index=idx)
    out.index.name = "date"

    cameo_cols = [f"cameo_{r}" for r in _INTERNAL_CAMEO]
    if not ev.empty:
        ev_f = ev[ev["EventRootCode"].isin(_INTERNAL_CAMEO)].copy()
        ev_f["date"] = pd.to_datetime(ev_f["date"])
        if not ev_f.empty:
            pivot = (ev_f.groupby([ev_f["date"].dt.date, "EventRootCode"]).size()
                          .unstack(fill_value=0)
                          .reindex(columns=_INTERNAL_CAMEO, fill_value=0))
            pivot.columns = [f"cameo_{c}" for c in pivot.columns]
            out = out.join(pivot, how="left")
    for c in cameo_cols:
        if c not in out.columns:
            out[c] = 0
        out[c] = out[c].fillna(0).astype(int)

    for theme in _INTERNAL_THEMES:
        out[theme] = 0
    out["gkg_neg_tone"] = 0.0
    out["n_docs"] = 0

    if not gkg.empty:
        g = gkg.copy()
        g["date"] = pd.to_datetime(g["date"]).dt.date
        g = g.set_index("date")
        common = out.index.intersection(g.index)
        if len(common):
            out.loc[common, "gkg_neg_tone"] = g.loc[common, "negative"].astype(float).fillna(0.0).values
            out.loc[common, "n_docs"] = g.loc[common, "n_docs"].astype(int).fillna(0).values
            for theme in _INTERNAL_THEMES:
                if theme in g.columns:
                    out.loc[common, theme] = g.loc[common, theme].astype(int).fillna(0).values

    out = out.reset_index().rename(columns={"index": "date"})
    out["date"] = pd.to_datetime(out["date"]).dt.date
    return out


def daily_bilateral_signals(target_name: str, date_from, date_to) -> pd.DataFrame:
    """Per-day series for the (Israel, target) dyad.

    Uses CAMEO 3-letter actor codes plus an ActionGeo fallback so we still capture
    events where one actor country code is null but the event location and the
    other actor identify the dyad.
    """
    anchor_meta = COUNTRIES[ANCHOR]
    target_meta = COUNTRIES[target_name]
    anchor_cameo = anchor_meta["cameo"]
    target_cameo = target_meta["cameo"]
    anchor_fips  = anchor_meta["fips"]
    target_fips  = target_meta["fips"]
    geo_extra    = _geo_clause(target_meta)

    cameo_aggs = ",\n  ".join([f"COUNTIF(EventRootCode = '{r}') AS cameo_{r}"
                                for r in _BILATERAL_CAMEO])
    sql = f"""
    SELECT
      DATE(_PARTITIONTIME) AS date,
      COUNT(*) AS n_events,
      {cameo_aggs},
      AVG(SAFE_CAST(AvgTone AS FLOAT64)) AS avgtone,
      AVG(SAFE_CAST(GoldsteinScale AS FLOAT64)) AS goldstein
    FROM `{EVENTS_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{date_from}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{date_to}'), INTERVAL 1 DAY))
      AND (
        -- A: both actors carry CAMEO country codes matching the dyad
        (Actor1CountryCode = '{anchor_cameo}' AND Actor2CountryCode = '{target_cameo}')
        OR (Actor1CountryCode = '{target_cameo}' AND Actor2CountryCode = '{anchor_cameo}')
        OR
        -- B: action took place in one country, the other actor matches
        (ActionGeo_CountryCode = '{anchor_fips}'
         AND (Actor1CountryCode = '{target_cameo}' OR Actor2CountryCode = '{target_cameo}'))
        OR (ActionGeo_CountryCode = '{target_fips}'
         AND (Actor1CountryCode = '{anchor_cameo}' OR Actor2CountryCode = '{anchor_cameo}'))
      )
      {geo_extra}
    GROUP BY date
    ORDER BY date
    """
    dry_run_mb(sql)
    ev = bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )

    idx = _daily_index(date_from, date_to)
    out = pd.DataFrame({"date": idx})
    if not ev.empty:
        ev["date"] = pd.to_datetime(ev["date"]).dt.date
        out = out.merge(ev, on="date", how="left")
    else:
        for r in _BILATERAL_CAMEO:
            out[f"cameo_{r}"] = 0
        out["n_events"] = 0
        out["avgtone"] = 0.0
        out["goldstein"] = 0.0
    out = out.fillna({c: 0 for c in out.columns if c.startswith("cameo_")})
    out = out.fillna({"n_events": 0, "avgtone": 0.0, "goldstein": 0.0})
    return out


print("Helper functions ready:")
print("  fetch_events(country, from, to)              -> events DataFrame")
print("  fetch_gkg(country, from, to)                 -> GKG daily counts + tone + theme hits")
print("  daily_internal_signals(country, from, to)    -> CAMEO + GKG signals per day per country")
print("  daily_bilateral_signals(target, from, to)    -> daily series for (Israel, target) dyad")
print("    bilateral uses CAMEO 3-letter actor codes (ISR/EGY/...) plus ActionGeo fallback")


## 8. Per-country internal instability

For each of the 8 entities, compare the **last 7 days** of internal-conflict signals against the **last 30 days**.

**Signals counted**:
- CAMEO root codes `14` (Protest), `17` (Coerce), `18` (Assault), `19` (Fight), `20` (Mass violence).
- GKG themes: water security, environmental water, infrastructure breakdown, power outage, cyber attack, rebellion, civil resistance, protest, governance reform demands.
- Mean negative-tone score from GKG `V2Tone`.

**Score** (intentionally simple - no machine learning, no logarithms):

$$\text{internal\_score} = \frac{\sum_s w_s \cdot \text{events}_\text{7d}(s)}{\sum_s w_s \cdot \text{events}_\text{30d}(s) + 1}$$

This is the **fraction of monthly weighted hostility that landed in the last 7 days**. Baseline value when nothing is heating up is `7 / 30 = 0.23`. A country at 0.40 means 40% of its monthly hostile-event total happened in the last week.

The output table shows the raw 7-day event count and the 30-day event count for every country, so the score can always be traced back to the underlying numbers.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

INTERNAL_WEIGHTS = {
    "cameo_20":      3.0,
    "cameo_19":      2.5,
    "cameo_18":      2.0,
    "cameo_17":      1.0,
    "cameo_14":      0.5,
    "water":         1.5,
    "electricity":   1.5,
    "civil_unrest":  1.0,
    "gkg_neg_tone":  1.0,
}
THEME_AGGREGATES = {
    "water":        ["WATER_SECURITY", "ENV_WATER"],
    "electricity":  ["POWER_OUTAGE", "INFRASTRUCTURE_BAD_ROADS", "CYBER_ATTACK"],
    "civil_unrest": ["REBELLION", "CIVIL_RESISTANCE", "PROTEST", "GOV_REQUIRES_REFORM"],
}


def _fetch_one_internal(country_name):
    df = daily_internal_signals(country_name, W30_FROM, TODAY).copy()
    df["country"] = country_name
    for agg, themes in THEME_AGGREGATES.items():
        cols = [c for c in themes if c in df.columns]
        df[agg] = df[cols].sum(axis=1) if cols else 0
    df["date"] = pd.to_datetime(df["date"]).dt.date
    return country_name, df


# Score every country (Israel still gets fetched so we have its data for other
# sections), but only TARGETS are ranked in the internal-instability output.
print(f"Fetching 30 days of internal signals for {len(COUNTRIES)} countries in parallel...")
_t0 = time.time()
_parts = []
with ThreadPoolExecutor(max_workers=len(COUNTRIES)) as _pool:
    _futs = {_pool.submit(_fetch_one_internal, c): c for c in COUNTRIES.keys()}
    for _fut in as_completed(_futs):
        _c = _futs[_fut]
        try:
            _cn, _df = _fut.result()
            print(f"  [OK] {_cn}")
            _parts.append(_df)
        except Exception as _e:
            print(f"  [FAIL] {_c}: {type(_e).__name__}: {_e}")
daily_internal_long = pd.concat(_parts, ignore_index=True)
print(f"Combined: {len(daily_internal_long):,} country-day rows in {time.time()-_t0:.1f}s.")

_rows = []
for country in TARGETS:   # <-- Israel excluded from ranking; only the 7 target dyads.
    sub = daily_internal_long[daily_internal_long["country"] == country]
    if sub.empty:
        _rows.append({"country": country, "share_score": np.nan, "volume_score_raw": np.nan,
                      "n_7d_events": 0, "n_30d_events": 0, "top_driver": None})
        continue
    w7  = sub[(sub["date"] >  W7_FROM)  & (sub["date"] <= TODAY)]
    w30 = sub[(sub["date"] >  W30_FROM) & (sub["date"] <= TODAY)]

    weighted_7d = 0.0
    weighted_30d = 0.0
    driver_contribs = {}
    for s, w in INTERNAL_WEIGHTS.items():
        if s not in sub.columns:
            continue
        e7  = float(w7[s].sum())
        e30 = float(w30[s].sum())
        weighted_7d  += w * e7
        weighted_30d += w * e30
        driver_contribs[s] = w * e7

    share_score = weighted_7d / (weighted_30d + 1.0)
    top_driver = max(driver_contribs, key=driver_contribs.get) if driver_contribs else None
    cameo_cols = [c for c in ["cameo_14","cameo_17","cameo_18","cameo_19","cameo_20"] if c in sub.columns]
    n7  = int(w7[cameo_cols].sum().sum())  if cameo_cols and not w7.empty  else 0
    n30 = int(w30[cameo_cols].sum().sum()) if cameo_cols and not w30.empty else 0
    _rows.append({"country": country, "share_score": float(share_score),
                  "volume_score_raw": float(weighted_7d),
                  "n_7d_events": n7, "n_30d_events": n30, "top_driver": top_driver})

internal_risk = pd.DataFrame(_rows).set_index("country").loc[TARGETS]

# Composite = sqrt(share * volume_norm) - geometric mean of "is heating up?" and
# "how much hostile news?". High share + high volume tops the chart; either weak
# alone is demoted.
_volraw = internal_risk["volume_score_raw"]
_vmax = float(_volraw.max()) if _volraw.max() and _volraw.max() > 0 else 1.0
internal_risk["volume_score"] = (_volraw / _vmax).fillna(0.0)
internal_risk["internal_score"] = np.sqrt(
    internal_risk["share_score"].fillna(0.0).clip(lower=0)
    * internal_risk["volume_score"].fillna(0.0).clip(lower=0)
)
internal_risk = internal_risk.sort_values("internal_score", ascending=False)

print("\nInternal instability ranking - 7 target countries (Israel excluded as anchor):\n")
display(internal_risk.style.format({
    "internal_score":   "{:.3f}",
    "share_score":      "{:.3f}",
    "volume_score":     "{:.3f}",
    "volume_score_raw": "{:,.0f}",
    "n_7d_events":      "{:,}",
    "n_30d_events":     "{:,}",
}))


In [ ]:
pdf = internal_risk.dropna(subset=["internal_score"]).sort_values("internal_score", ascending=True)
top_country = pdf["internal_score"].idxmax() if not pdf.empty else None
colors = ["#c0392b" if c == top_country else "#4a7ab8" for c in pdf.index]

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.barh(pdf.index, pdf["internal_score"], color=colors, edgecolor="black", linewidth=0.4)
ax.set_xlabel("Internal instability composite score  =  sqrt(share x volume)")
ax.set_title(f"Per-country internal instability  ({TODAY.isoformat()})")
ax.grid(False)

# Stretch axis so annotations fit inside the frame
_xmax = float(pdf["internal_score"].max()) if not pdf.empty else 1.0
ax.set_xlim(0, max(_xmax * 1.75, 0.8))

for bar, country in zip(bars, pdf.index):
    val = pdf.loc[country, "internal_score"]
    n7 = int(internal_risk.loc[country, "n_7d_events"])
    n30 = int(internal_risk.loc[country, "n_30d_events"])
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"  score={val:.2f}   7d={n7:,}   30d={n30:,}",
            va="center", ha="left", fontsize=9)

plt.tight_layout()
Path("assets").mkdir(exist_ok=True)
fig.savefig("assets/viz_internal_instability.png", dpi=140, bbox_inches="tight")
plt.show()


## 9. Bilateral escalation pressure - Israel vs each X

For each `(Israel, X)` pair we pull every event in the last 30 days where one party is Israel and the other is the target country. We weight hostile events by severity (mass-violence events count more than threats) and ask:

> Where does this dyad rank against the others on absolute weighted hostile coverage this week?

$$\text{raw}(X) = \sum_s w_s \cdot \text{events}_\text{7d}(s)$$

$$\text{score}(X) = \frac{\text{raw}(X) - \min_X \text{raw}(X)}{\max_X \text{raw}(X) - \min_X \text{raw}(X)}$$

Min-max scaled across the seven dyads so 1.00 always means *the dyad with the most hostile coverage this week*, 0.00 means *the dyad with the least*. Higher score = more absolute hostile news. The score is shown alongside the raw 7-day and 30-day event counts so the ranking is fully auditable.

**Signals counted**: CAMEO root codes 13 (threaten), 15 (force posture / drills), 17 (coerce), 18 (assault), 19 (fight), 20 (mass violence).

In [ ]:
BILATERAL_WEIGHTS = {
    "cameo_20": 3.0,
    "cameo_19": 2.5,
    "cameo_18": 2.0,
    "cameo_15": 1.5,
    "cameo_17": 1.0,
    "cameo_13": 0.8,
}


def _fetch_bilateral(target_name):
    df = daily_bilateral_signals(target_name, W30_FROM, TODAY).copy()
    df["date"] = pd.to_datetime(df["date"]).dt.date
    return target_name, df


print(f"Fetching 30 days of bilateral signals for {len(TARGETS)} dyads in parallel...")
_t0 = time.time()
_pair_data = {}
with ThreadPoolExecutor(max_workers=len(TARGETS)) as _pool:
    _futs = {_pool.submit(_fetch_bilateral, t): t for t in TARGETS}
    for _fut in as_completed(_futs):
        _t = _futs[_fut]
        try:
            _tn, _df = _fut.result()
            print(f"  [OK] Israel <-> {_tn}")
            _pair_data[_tn] = _df
        except Exception as _e:
            print(f"  [FAIL] Israel <-> {_t}: {type(_e).__name__}: {_e}")
            _pair_data[_t] = None
print(f"Fetched in {time.time()-_t0:.1f}s.")

_rows = []
for target in TARGETS:
    df = _pair_data.get(target)
    if df is None or df.empty:
        _rows.append({"target": target, "raw": np.nan,
                      "n_7d_events": 0, "n_30d_events": 0, "top_driver": None})
        continue
    w7  = df[(df["date"] >  W7_FROM)  & (df["date"] <= TODAY)]
    w30 = df[(df["date"] >  W30_FROM) & (df["date"] <= TODAY)]

    raw = 0.0
    driver_contribs = {}
    for s, w in BILATERAL_WEIGHTS.items():
        if s not in df.columns:
            continue
        e7 = float(w7[s].sum())
        raw += w * e7
        driver_contribs[s] = w * e7

    top_driver = max(driver_contribs, key=driver_contribs.get) if driver_contribs else None
    cameo_cols = [c for c in df.columns if c.startswith("cameo_")]
    n7  = int(w7[cameo_cols].sum().sum())  if cameo_cols and not w7.empty  else 0
    n30 = int(w30[cameo_cols].sum().sum()) if cameo_cols and not w30.empty else 0
    _rows.append({"target": target, "raw": float(raw),
                  "n_7d_events": n7, "n_30d_events": n30,
                  "top_driver": top_driver})

bilateral_risk = pd.DataFrame(_rows).set_index("target")

_raws = bilateral_risk["raw"].dropna()
if len(_raws) > 1 and _raws.max() > _raws.min():
    bilateral_risk["score"] = (bilateral_risk["raw"] - _raws.min()) / (_raws.max() - _raws.min())
else:
    bilateral_risk["score"] = 0.0
bilateral_risk = bilateral_risk.sort_values("score", ascending=False)

print("\nBilateral escalation pressure - severity-weighted recent volume, min-max scaled:\n")
display(bilateral_risk[["score", "raw", "n_7d_events", "n_30d_events", "top_driver"]].style.format({
    "score":         "{:.3f}",
    "raw":           "{:,.1f}",
    "n_7d_events":   "{:,}",
    "n_30d_events":  "{:,}",
}))


In [ ]:
pdf = bilateral_risk.dropna(subset=["score"]).sort_values("score", ascending=True)
top = pdf["score"].idxmax() if not pdf.empty else None
colors = ["#c0392b" if t == top else "#4a7ab8" for t in pdf.index]

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.barh(pdf.index, pdf["score"], color=colors, edgecolor="black", linewidth=0.4)
ax.set_xlim(0, 1.4)
ax.set_xlabel("Bilateral escalation pressure  (min-max severity-weighted 7-day volume)")
ax.set_title(f"Bilateral escalation pressure - Israel vs each X  ({TODAY.isoformat()})")
ax.grid(False)

for bar, t in zip(bars, pdf.index):
    val = pdf.loc[t, "score"]
    n7  = int(bilateral_risk.loc[t, "n_7d_events"])
    n30 = int(bilateral_risk.loc[t, "n_30d_events"])
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"  score={val:.2f}   7d={n7:,}   30d={n30:,}",
            va="center", ha="left", fontsize=9)

plt.tight_layout()
Path("assets").mkdir(exist_ok=True)
fig.savefig("assets/viz_bilateral_pressure.png", dpi=140, bbox_inches="tight")
plt.show()


## 10. Correlation, covariance, and daily hostility time series

Each row in the daily-hostility matrix is one day in the last 30. Each column is the daily count of CAMEO 18 (assault) + 19 (fight) + 20 (mass violence) events for one `(Israel, X)` dyad. From it we compute:

- **Pearson correlation matrix (7 x 7, green palette)**: do these dyads' news cycles rise and fall on the same days? A high off-diagonal value means "Israel-A spikes the same days as Israel-B".
- **Covariance matrix (7 x 7, green palette)**: same shape but unstandardised. The magnitudes reflect raw event-count variance, not just direction.
- **Co-movement score**: each dyad's mean correlation with the other six. The dyad with the highest co-movement is the *canary* - its bilateral signal is the closest proxy for the regional pattern as a whole.
- **Daily hostile-event counts of Israel vs each X** - the underlying time series. This is the chart you should look at first; the correlation and covariance numbers are summaries of *this*.

In [ ]:
HOSTILE_CAMEOS = ["cameo_18", "cameo_19", "cameo_20"]

_full_idx = pd.date_range(W30_FROM, TODAY, freq="D").date
_series_by_target = {}
for _tgt in TARGETS:
    _df = _pair_data.get(_tgt)
    if _df is None or _df.empty:
        _series_by_target[_tgt] = pd.Series(0, index=_full_idx)
        continue
    _avail = [c for c in HOSTILE_CAMEOS if c in _df.columns]
    if not _avail:
        _series_by_target[_tgt] = pd.Series(0, index=_full_idx)
        continue
    _daily = (_df.assign(hostility=_df[_avail].sum(axis=1))
                .groupby("date")["hostility"].sum())
    _series_by_target[_tgt] = _daily.reindex(_full_idx, fill_value=0).astype(float)

dyad_matrix = pd.DataFrame(_series_by_target, index=_full_idx)
dyad_matrix.index.name = "date"

corr = dyad_matrix.corr(method="pearson").fillna(0.0)
cov  = dyad_matrix.cov().fillna(0.0)

_off_diag = corr.where(~np.eye(len(corr), dtype=bool))
comovement = _off_diag.mean(axis=1).sort_values(ascending=False)
canary = comovement.idxmax() if len(comovement) else None

print("Mean correlation with the other 6 dyads (co-movement score):")
for _t, _v in comovement.items():
    print(f"  {_t:<12} {_v:+.3f}")
print(f"\nCanary  (highest co-movement, i.e. its signal most resembles the regional whole): {canary}")

# Daily hostile-event time series chart - one line per dyad - look at this FIRST
fig, ax = plt.subplots(figsize=(13, 5.5))
_palette = plt.get_cmap("tab10")
for _i, _tgt in enumerate(TARGETS):
    ax.plot(dyad_matrix.index, dyad_matrix[_tgt],
            label=_tgt, lw=1.8, color=_palette(_i % 10))
ax.set_title(f"Daily hostile-event count  -  Israel  vs each target country  ({W30_FROM} to {TODAY})")
ax.set_xlabel("date")
ax.set_ylabel("hostile events / day (CAMEO 18 + 19 + 20)")
ax.grid(False)
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", borderaxespad=0., fontsize=9)
plt.tight_layout()
Path("assets").mkdir(exist_ok=True)
fig.savefig("assets/viz_daily_hostile_events.png", dpi=140, bbox_inches="tight")
plt.show()

# Side-by-side heatmaps - green palette
green_cmap = sns.light_palette("seagreen", as_cmap=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(corr, ax=axes[0], cmap=green_cmap, vmin=-1, vmax=1,
            annot=True, fmt=".2f", linewidths=0.4, square=True,
            cbar_kws={"shrink": 0.7, "label": "Pearson r"})
axes[0].set_title(f"Pearson correlation  ({W30_FROM} to {TODAY})")

sns.heatmap(cov, ax=axes[1], cmap=green_cmap,
            annot=True, fmt=".1f", linewidths=0.4, square=True,
            cbar_kws={"shrink": 0.7, "label": "covariance"})
axes[1].set_title("Covariance (unstandardised)")
fig.tight_layout()
fig.savefig("assets/viz_corr_cov.png", dpi=140, bbox_inches="tight")
plt.show()

# Top dyad pairs
_pairs = []
_tcols = list(corr.columns)
for _i in range(len(_tcols)):
    for _j in range(_i + 1, len(_tcols)):
        _pairs.append((_tcols[_i], _tcols[_j], corr.iloc[_i, _j]))
top_pairs = pd.DataFrame(_pairs, columns=["A", "B", "pearson_r"]).sort_values("pearson_r", ascending=False)
print("\nTop dyad pairs whose Israel-bilateral series move together:")
display(top_pairs.head(7).style.format({"pearson_r": "{:.3f}"}))


## 11. Combined risk dashboard  (interactive)

Interactive scatter using Plotly. Each marker is one target. The x-axis is the internal instability score from section 8; the y-axis is the bilateral pressure score from section 9. Marker size = total hostile event count in the last 7 days, marker colour = co-movement score from section 11.

**Hover** over any marker to see all the supporting numbers without labels overlapping.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

_rows = []
for target in TARGETS:
    _rows.append({
        "target": target,
        "internal_score": float(internal_risk.loc[target, "internal_score"])
            if target in internal_risk.index else np.nan,
        "internal_top_driver": internal_risk.loc[target, "top_driver"]
            if target in internal_risk.index else None,
        "internal_7d_events": int(internal_risk.loc[target, "n_7d_events"])
            if target in internal_risk.index else 0,
        "bilateral_score": float(bilateral_risk.loc[target, "score"])
            if target in bilateral_risk.index else np.nan,
        "bilateral_top_driver": bilateral_risk.loc[target, "top_driver"]
            if target in bilateral_risk.index else None,
        "n_7d_events": int(bilateral_risk.loc[target, "n_7d_events"])
            if target in bilateral_risk.index else 0,
        "n_30d_events": int(bilateral_risk.loc[target, "n_30d_events"])
            if target in bilateral_risk.index else 0,
        "comovement": float(comovement.get(target, np.nan)),
    })
dashboard = pd.DataFrame(_rows)

_sizes = (dashboard["n_7d_events"].clip(lower=0).astype(float))
_sizes = 14 + 60 * np.sqrt(_sizes / (max(_sizes.max(), 1.0)))

_fig = go.Figure()
_fig.add_trace(go.Scatter(
    x=dashboard["internal_score"], y=dashboard["bilateral_score"],
    mode="markers+text",
    text=dashboard["target"], textposition="top center", textfont=dict(size=11),
    marker=dict(
        size=_sizes, color=dashboard["comovement"],
        colorscale="Greens", showscale=True,
        colorbar=dict(title="co-movement", thickness=12, len=0.6),
        line=dict(color="black", width=0.6),
    ),
    customdata=np.stack([
        dashboard["target"],
        dashboard["bilateral_score"].round(3),
        dashboard["n_7d_events"], dashboard["n_30d_events"],
        dashboard["internal_score"].round(3),
        dashboard["internal_7d_events"],
        dashboard["comovement"].round(3),
        dashboard["bilateral_top_driver"].fillna("-"),
        dashboard["internal_top_driver"].fillna("-"),
    ], axis=-1),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "bilateral pressure  : %{customdata[1]}<br>"
        "bilateral 7d events : %{customdata[2]}<br>"
        "bilateral 30d events: %{customdata[3]}<br>"
        "internal score      : %{customdata[4]}<br>"
        "internal 7d events  : %{customdata[5]}<br>"
        "co-movement         : %{customdata[6]}<br>"
        "bilateral driver    : %{customdata[7]}<br>"
        "internal driver     : %{customdata[8]}<extra></extra>"
    ),
))
_fig.add_vline(x=dashboard["internal_score"].median(), line_dash="dash",
               line_color="grey", line_width=1)
_fig.add_hline(y=dashboard["bilateral_score"].median(), line_dash="dash",
               line_color="grey", line_width=1)
_fig.update_layout(
    title=f"Combined risk dashboard - Israel vs all targets ({TODAY.isoformat()})",
    xaxis_title="Internal instability composite",
    yaxis_title="Bilateral escalation pressure",
    width=950, height=620, template="simple_white",
    showlegend=False,
)
_fig.update_xaxes(showgrid=False)
_fig.update_yaxes(showgrid=False, range=[0, 1])
_fig.show()

# Static PNG export for the README (requires kaleido).
try:
    Path("assets").mkdir(exist_ok=True)
    _fig.write_image("assets/viz_combined_dashboard.png", width=1100, height=720, scale=2)
except Exception as _e:
    print(f"PNG export of dashboard skipped: {type(_e).__name__}: {_e}")
    print("Install kaleido to enable: %pip install -q kaleido")

ranked = dashboard.sort_values(
    by=["bilateral_score", "internal_score", "comovement"],
    ascending=[False, False, False], na_position="last",
).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

print("\nFinal ranking - sorted by bilateral pressure, then internal score, then co-movement:\n")
display(ranked.style.format({
    "internal_score": "{:.3f}",
    "bilateral_score": "{:.3f}",
    "comovement": "{:+.3f}",
    "n_7d_events": "{:,}",
    "n_30d_events": "{:,}",
    "internal_7d_events": "{:,}",
}))

if not ranked.empty:
    top = ranked.iloc[0]
    print(f"\nHeadline: closest to escalation with {ANCHOR} -> {top['target']}  "
          f"(bilateral={top['bilateral_score']:.2f}, "
          f"internal={top['internal_score']:.2f}, "
          f"7d events={top['n_7d_events']:,})")


## 12. GKG knowledge graph: entity co-occurrence network

GDELT's Global Knowledge Graph extracts named entities (persons, organizations) from every news document it processes. Below we pull, for each country, the top 10 organizations and persons that co-appear in GKG documents about that country over the last 30 days, then render the whole thing as a network: countries are red nodes, top entities are blue, edges are weighted by co-mention count.

What you're looking at: **the actual named figures driving each country's news in the last month**.

In [ ]:
import networkx as nx
import re

ENTITY_TOP_N = 10
ENTITY_STOP = {"israel", "egypt", "turkey", "jordan", "iran", "lebanon", "gaza",
               "west bank", "israeli", "palestinian", "palestinians", "israelis",
               "egyptian", "turkish", "jordanian", "iranian", "lebanese",
               "united states", "us", "usa", "united nations", "un",
               "european union", "eu", "reuters", "ap"}


def _fetch_entities(country_name):
    meta = COUNTRIES[country_name]
    sql = f"""
    SELECT V2Persons, V2Organizations
    FROM `{GKG_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{W30_FROM}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{TODAY}'), INTERVAL 1 DAY))
      AND V2Locations LIKE '%#{meta["fips"]}#%'
    """
    dry_run_mb(sql)
    return country_name, bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )


print(f"Fetching GKG entities for {len(COUNTRIES)} countries in parallel...")
_t0 = time.time()
_entity_data = {}
with ThreadPoolExecutor(max_workers=len(COUNTRIES)) as _pool:
    _futs = {_pool.submit(_fetch_entities, c): c for c in COUNTRIES.keys()}
    for _fut in as_completed(_futs):
        try:
            _cn, _df = _fut.result()
            _entity_data[_cn] = _df
            print(f"  [OK] {_cn}: {len(_df):,} docs")
        except Exception as _e:
            print(f"  [FAIL] {_futs[_fut]}: {type(_e).__name__}: {_e}")
print(f"Fetched in {time.time()-_t0:.1f}s.")

from collections import Counter

def _explode(s, kind="orgs"):
    if not isinstance(s, str): return []
    items = []
    for tok in s.split(";"):
        name = tok.split(",")[0].strip().lower()
        if not name or name in ENTITY_STOP: continue
        if len(name) < 3 or len(name) > 50: continue
        if not re.search(r"[a-z]", name): continue
        items.append(name)
    return items

country_top_entities = {}
for country, df in _entity_data.items():
    cnt = Counter()
    if df is None or df.empty:
        country_top_entities[country] = []
        continue
    for col in ("V2Persons", "V2Organizations"):
        if col not in df.columns: continue
        for raw in df[col].dropna():
            for name in _explode(raw):
                cnt[name] += 1
    country_top_entities[country] = cnt.most_common(ENTITY_TOP_N)

# Build the network
G = nx.Graph()
for country, entities in country_top_entities.items():
    G.add_node(country, kind="country")
    for entity, weight in entities:
        if not G.has_node(entity):
            G.add_node(entity, kind="entity")
        G.add_edge(country, entity, weight=weight)

print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

if G.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(14, 11))
    pos = nx.spring_layout(G, seed=42, k=1.2, iterations=80)
    country_nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == "country"]
    entity_nodes  = [n for n, d in G.nodes(data=True) if d.get("kind") == "entity"]

    nx.draw_networkx_nodes(G, pos, nodelist=country_nodes, node_color="#c0392b",
                            node_size=1600, edgecolors="black", linewidths=1.2, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=entity_nodes, node_color="#4a7ab8",
                            node_size=420, edgecolors="black", linewidths=0.4, ax=ax)
    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
    if edge_weights:
        max_w = max(edge_weights)
        widths = [0.3 + 2.0 * (w / max_w) for w in edge_weights]
    else:
        widths = 0.5
    nx.draw_networkx_edges(G, pos, width=widths, edge_color="#888888", alpha=0.55, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={n: n.title() for n in country_nodes},
                             font_size=11, font_weight="bold", ax=ax)
    nx.draw_networkx_labels(G, pos, labels={n: n.title() for n in entity_nodes},
                             font_size=7, ax=ax)
    ax.set_title(f"GKG entity co-occurrence network  ({W30_FROM} to {TODAY})", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    Path("assets").mkdir(exist_ok=True)
    fig.savefig("assets/viz_gkg_entity.png", dpi=140, bbox_inches="tight")
    plt.show()

print("\nTop entities per country (from GKG V2Persons + V2Organizations):")
for country, entities in country_top_entities.items():
    if entities:
        names = ", ".join([f"{name} ({w})" for name, w in entities[:5]])
        print(f"  {country}: {names}")
    else:
        print(f"  {country}: (no entities)")


## 13. Per-country word clouds  (general events)

For each of the eight target countries, build a word cloud of the most-mentioned actor names (`Actor1Name + Actor2Name`) weighted by `NumMentions` over the last 30 days. Surfaces the named figures driving each country's news on its own, independent of any Israel link.

In [ ]:
from wordcloud import WordCloud
from collections import Counter

_STOP = {
    "UNKNOWN", "GOVERNMENT", "OFFICIAL", "REBEL", "CIVILIAN", "MILITANT",
    "NONE", "", "INTERNATIONAL", "PARLIAMENT", "MINISTER", "MILITARY",
    "WORLD", "PEOPLE", "FOREIGN", "WOMAN", "MAN", "REGION", "GROUP",
    "POLICE", "PRESIDENT", "MEDIA", "CITIZEN", "WORKER", "EMPLOYEE",
}

def _per_country_actor_sql(country_name):
    meta = COUNTRIES[country_name]
    geo_extra = _geo_clause(meta)
    return f"""
    SELECT Actor1Name, Actor2Name, CAST(NumMentions AS INT64) AS NumMentions
    FROM `{EVENTS_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{W30_FROM}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{TODAY}'), INTERVAL 1 DAY))
      AND ActionGeo_CountryCode = '{meta["fips"]}'
      AND NumMentions > 0
      AND (Actor1Name IS NOT NULL OR Actor2Name IS NOT NULL)
      {geo_extra}
    """

def _fetch_country_actors(country_name):
    sql = _per_country_actor_sql(country_name)
    dry_run_mb(sql)
    return country_name, bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )

print(f"Fetching actor names for {len(COUNTRIES)} countries in parallel...")
_t0 = time.time()
_country_actors = {}
with ThreadPoolExecutor(max_workers=len(COUNTRIES)) as _pool:
    _futs = {_pool.submit(_fetch_country_actors, c): c for c in COUNTRIES.keys()}
    for _fut in as_completed(_futs):
        try:
            _cn, _df = _fut.result()
            _country_actors[_cn] = _df
            print(f"  [OK] {_cn}: {len(_df):,} rows")
        except Exception as _e:
            print(f"  [FAIL] {_futs[_fut]}: {type(_e).__name__}: {_e}")
print(f"Fetched in {time.time()-_t0:.1f}s.")

def _build_actor_counter(df):
    cnt = Counter()
    if df is None or df.empty: return cnt
    for col in ("Actor1Name", "Actor2Name"):
        if col not in df.columns: continue
        slc = df[[col, "NumMentions"]].dropna()
        slc = slc[~slc[col].str.upper().isin(_STOP)]
        for name, n in zip(slc[col], slc["NumMentions"].astype(int)):
            cnt[str(name).strip().upper()] += int(n)
    return cnt

fig, axes = plt.subplots(4, 2, figsize=(15, 17))
for ax, country in zip(axes.flat, COUNTRIES.keys()):
    cnt = _build_actor_counter(_country_actors.get(country))
    if not cnt:
        ax.text(0.5, 0.5, f"{country}\nno actor data", ha="center", va="center", fontsize=12)
        ax.axis("off"); continue
    wc = WordCloud(width=900, height=500, background_color="white",
                   colormap="viridis", max_words=80, prefer_horizontal=0.9
                   ).generate_from_frequencies(dict(cnt))
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{country}  ({sum(cnt.values()):,} mentions)", fontsize=11)
    ax.axis("off")
fig.suptitle(f"Top actors per country  (general events, {W30_FROM} -> {TODAY})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


## 14. Per-country word clouds  (bilateral conflict signal vs Israel)

Now narrow to just the *bilateral* event stream `(Israel <-> X)`. Same actor extraction, but only on the events where one party is Israel and the other is the target. Surfaces the named figures driving each country's friction or cooperation with Israel.

In [ ]:
def _bilateral_actor_sql(target_name):
    meta = COUNTRIES[target_name]
    anchor_cameo = COUNTRIES[ANCHOR]["cameo"]
    target_cameo = meta["cameo"]
    anchor_fips  = COUNTRIES[ANCHOR]["fips"]
    target_fips  = meta["fips"]
    geo_extra    = _geo_clause(meta)
    return f"""
    SELECT Actor1Name, Actor2Name, CAST(NumMentions AS INT64) AS NumMentions
    FROM `{EVENTS_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP('{W30_FROM}')
      AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE('{TODAY}'), INTERVAL 1 DAY))
      AND NumMentions > 0
      AND (Actor1Name IS NOT NULL OR Actor2Name IS NOT NULL)
      AND (
        (Actor1CountryCode = '{anchor_cameo}' AND Actor2CountryCode = '{target_cameo}')
        OR (Actor1CountryCode = '{target_cameo}' AND Actor2CountryCode = '{anchor_cameo}')
        OR (ActionGeo_CountryCode = '{anchor_fips}'
            AND (Actor1CountryCode = '{target_cameo}' OR Actor2CountryCode = '{target_cameo}'))
        OR (ActionGeo_CountryCode = '{target_fips}'
            AND (Actor1CountryCode = '{anchor_cameo}' OR Actor2CountryCode = '{anchor_cameo}'))
      )
      {geo_extra}
    """

def _fetch_bilateral_actors(target_name):
    sql = _bilateral_actor_sql(target_name)
    dry_run_mb(sql)
    return target_name, bq.query(sql).result().to_dataframe(
        create_bqstorage_client=True, progress_bar_type=None,
    )

print(f"Fetching bilateral actor names for {len(TARGETS)} dyads in parallel...")
_t0 = time.time()
_bilateral_actors = {}
with ThreadPoolExecutor(max_workers=len(TARGETS)) as _pool:
    _futs = {_pool.submit(_fetch_bilateral_actors, t): t for t in TARGETS}
    for _fut in as_completed(_futs):
        try:
            _tn, _df = _fut.result()
            _bilateral_actors[_tn] = _df
            print(f"  [OK] Israel <-> {_tn}: {len(_df):,} rows")
        except Exception as _e:
            print(f"  [FAIL] Israel <-> {_futs[_fut]}: {type(_e).__name__}: {_e}")
print(f"Fetched in {time.time()-_t0:.1f}s.")

fig, axes = plt.subplots(4, 2, figsize=(15, 17))
_plot_targets = TARGETS + [None]  # 8 panels; last is blank to fill 4x2
for ax, target in zip(axes.flat, _plot_targets):
    if target is None:
        ax.axis("off"); continue
    cnt = _build_actor_counter(_bilateral_actors.get(target))
    if not cnt:
        ax.text(0.5, 0.5, f"Israel <-> {target}\nno bilateral actor data",
                ha="center", va="center", fontsize=12)
        ax.axis("off"); continue
    wc = WordCloud(width=900, height=500, background_color="white",
                   colormap="plasma", max_words=80, prefer_horizontal=0.9
                   ).generate_from_frequencies(dict(cnt))
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Israel <-> {target}  ({sum(cnt.values()):,} mentions)", fontsize=11)
    ax.axis("off")
fig.suptitle(f"Top actors per bilateral dyad  (Israel <-> X, {W30_FROM} -> {TODAY})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


## 15. Per-country 7d vs 30d time-series  (how news is evolving)

For each country, plot the daily count of hostile-coded events (CAMEO 18 + 19 + 20) over the last 30 days. The trailing 7-day window is shaded so you can see whether the recent pulse is above or below the monthly trend. Headline numbers (7-day and 30-day raw totals) are printed above each panel.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 13), sharex=False)
_hostile_cols = ["cameo_18", "cameo_19", "cameo_20"]
for ax, country in zip(axes.flat, COUNTRIES.keys()):
    sub = daily_internal_long[daily_internal_long["country"] == country].copy()
    if sub.empty:
        ax.text(0.5, 0.5, f"{country}\nno data", ha="center", va="center", fontsize=11)
        ax.axis("off"); continue
    sub = sub.sort_values("date")
    avail = [c for c in _hostile_cols if c in sub.columns]
    if not avail:
        ax.text(0.5, 0.5, f"{country}\nno hostile columns", ha="center", va="center", fontsize=11)
        ax.axis("off"); continue
    daily = sub[avail].sum(axis=1)
    sub["daily_hostility"] = daily.values
    n7  = int(sub.loc[(sub["date"] > W7_FROM)  & (sub["date"] <= TODAY),  "daily_hostility"].sum())
    n30 = int(sub.loc[(sub["date"] > W30_FROM) & (sub["date"] <= TODAY), "daily_hostility"].sum())
    ax.plot(pd.to_datetime(sub["date"]), sub["daily_hostility"],
            color="#c0392b", lw=1.6, marker="o", markersize=3)
    ax.axvspan(pd.to_datetime(W7_FROM), pd.to_datetime(TODAY),
               color="#c0392b", alpha=0.10, label="last 7 days")
    ax.set_title(f"{country}  -  7d total: {n7:,}  /  30d total: {n30:,}", fontsize=11)
    ax.set_ylabel("hostile events / day", fontsize=9)
    ax.grid(False)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    if country == list(COUNTRIES.keys())[0]:
        ax.legend(loc="upper left", fontsize=8)
fig.suptitle(f"Per-country daily hostile-event count  ({W30_FROM} to {TODAY})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## 16. Limitations and responsible use

Read these before using any number from this notebook for any decision.

- **The headline number is not P(war).** It is a media-coverage acceleration index. It tells you which dyad is heating up fastest in the news; it does not say a war is actually more or less likely on any calendar.
- **No training, no calibration.** The weights in the scoring formula were chosen by hand. Two reasonable people picking different weights would get different rankings.
- **Reporting bias is real.** GDELT is built from worldwide news media. Events outside English / major-language coverage are under-counted. A spike in this notebook can be a real escalation or a spike in *coverage* (anniversary stories, UN speeches, a single viral incident).
- **Correlation is not causation.** Media coverage and on-the-ground events share underlying news cycles. Acceleration in Israel-vs-X coverage does not prove preparation for war.
- **Bolt-from-the-blue events are invisible to a 7-day window.** The score ramps up in the days *after* a surprise attack, not before. Recent examples in the GDELT 2.0 era (Oct 7 2023 morning, Apr 13 2024 Iran first strike) would have been below the warning threshold the day before they happened.
- **Gaza and West Bank share FIPS country code `WE`.** The dyad helpers disambiguate by `ActionGeo_FullName` substring match - this is imperfect; some events tagged only at country level may be misallocated.
- **Do not attribute named individuals.** The actor and GKG entity outputs aggregate counts only; never use them to make claims about specific people.

**GDELT** is the source for everything in this notebook. Project page: <https://www.gdeltproject.org/>. The Events 2.0 table is `gdelt-bq.gdeltv2.events_partitioned`; the GKG 2.0 table is `gdelt-bq.gdeltv2.gkg_partitioned`. Both are public datasets you read at your own cost via BigQuery.